In [1]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from src import Questions
import matplotlib.patches as mpatches
from scipy.stats import ttest_ind
from itertools import combinations

In [2]:
# definitions
models = [
    "gpt_4o",
    "llama31_70B_Inst",
    "Qwen25_72B_Inst",
    "llama31_8B_Inst",
    "Qwen25_7B_Inst",
    "llama32_1B_Inst",
    "Qwen25_1dot5B_Inst"
]
task_types = [
    'mt_and_summarization',
    'syntaxes',
    'knowledge_and_reasoning',
    'semantics',
    'qa_extraction'
]
demography_profiles = [
    "employment_sector",
    "race",
    "education",
    "sex",
    "age",
    "language",
    "nationality_group"
]
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"
EXCLUDED_SCORES = ["2_13", "23_", "24_", "25_", "27_"]

questions = Questions().get_questions()

stacked_model_df = []
for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]
    stacked_model_df.append(score_df[task_cols])

def get_min_max_from_stacked_df(stacked_model_df):
    # Stack all DataFrames into one
    combined_df = pd.concat(stacked_model_df, ignore_index=True)

    # Compute min and max for each column
    return {
        col: {
            'min': combined_df[col].min(),
            'max': combined_df[col].max()
        }
        for col in combined_df.columns
    }

universal_scoredf_minmax = get_min_max_from_stacked_df(stacked_model_df)

scorelists_by_model = {}
for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]

    fullscores_per_dp = {}
    for dp in demography_profiles:
        fullscores_per_dp[dp] = {}
        for tc in task_cols:
            fullscores_per_dp[dp][tc] = {}
            if not any(tc.startswith(prefix) for prefix in EXCLUDED_SCORES):
                demo_grouped = score_df.groupby([dp])
                for demo_name, group in demo_grouped:
                    # implement here for exceptions
                    if dp == "employment_sector":
                        group = group[group["language"] == "L1"]
                    elif dp == "education":
                        group = group[group["language"] == "L1"]
                    elif dp == "sex":
                        group = group[group["language"] == "L1"]
                        
                    elif(demo_name == "Black"):
                        group = group[group["language"] == "L1"]
                    elif(demo_name == "White"):
                        group = group[group["language"] == "L1"]
                    # implement here for exceptions
                    fullscores_per_dp[dp][tc][demo_name[0]] = group[tc].dropna().to_list()
    scorelists_by_model[model] = fullscores_per_dp


In [15]:
REJECT_THRESHOLD = 0.05
from scipy.stats import f_oneway

def compute_anova_score(data_dict):
    # Extract the lists of scores
    groups = list(data_dict.values())
    # Perform one-way ANOVA
    f_statistic, p_value = f_oneway(*groups)
    return f_statistic, p_value


for modl in models:
    for k in scorelists_by_model[modl].keys():
        tasks = [qdi for qdi in scorelists_by_model[modl][k] if not any(qdi.startswith(prefix) for prefix in EXCLUDED_SCORES)]
        task_count = len(tasks)
        for qdi in tasks:
            f, p = compute_anova_score(scorelists_by_model[modl][k][qdi])
            adj_p = p * task_count
            if (adj_p < REJECT_THRESHOLD):
                print(modl, "REJECTED!", f"task-{qdi}", k, adj_p)

gpt_4o REJECTED! task-15_0 age 0.004026145028783612
llama31_70B_Inst REJECTED! task-28_0 age 0.014737161360797392
llama31_70B_Inst REJECTED! task-20_0 language 0.033048407179910026
llama31_8B_Inst REJECTED! task-21_0 race 0.003965716165915619
llama31_8B_Inst REJECTED! task-21_0 age 0.0008923700941678354
llama31_8B_Inst REJECTED! task-21_0 language 0.001348967116443219
llama31_8B_Inst REJECTED! task-21_0 nationality_group 6.58619994050562e-05
Qwen25_1dot5B_Inst REJECTED! task-20_0 language 0.0021436674777911475
Qwen25_1dot5B_Inst REJECTED! task-20_0 nationality_group 0.0025613624533352853


In [4]:
# naive p values

def pairwise_p_values(data_dict):
    results = {}
    for (key1, values1), (key2, values2) in combinations(data_dict.items(), 2):
        stat, p = ttest_ind(values1, values2, equal_var=False)
        results[f"{key1} ---- {key2}"] = p
    return results

def calculate_diversity_score(list_of_values):
    if len(list_of_values) < 2:
        return 0  # No diversity if less than 2 elements
    
    total = 0
    count = 0
    for i in range(len(list_of_values)):
        for j in range(i + 1, len(list_of_values)):
            total += abs(list_of_values[i] - list_of_values[j])
            count += 1
    return total / count

REJECT_THRESHOLD = 0.05

model_pvalues = []
model_significant_diversity_scores = []
for model, all_scores in scorelists_by_model.items():
    dp_unfairness = {}
    dp_diversity = {}
    for dp, dp_values in all_scores.items():
        unfair_tasks = []
        diversity_scores_of_unfair_tasks = []
        for qdi, qdi_values in dp_values.items():
            if not any(qdi.startswith(prefix) for prefix in EXCLUDED_SCORES):
                this_pair_unfair = False
                pvalue_pairs = pairwise_p_values(qdi_values)
                qdi_min_val = universal_scoredf_minmax[qdi]['min']
                qdi_max_val = universal_scoredf_minmax[qdi]['max']
                for pairname, pairpval in pvalue_pairs.items():
                    if pairpval < REJECT_THRESHOLD:
                        this_pair_unfair = True
                        pair1, pair2 = pairname.split(" ---- ")
                    
                        qdi_val_pair1 = [
                            (x - qdi_min_val) / (qdi_max_val - qdi_min_val)
                            for x in qdi_values[pair1]
                        ]
                        
                        qdi_val_pair2 = [
                            (x - qdi_min_val) / (qdi_max_val - qdi_min_val)
                            for x in qdi_values[pair2]
                        ]
                        diversity_scores_of_unfair_tasks.append(abs(np.mean(qdi_val_pair1) - np.mean(qdi_val_pair2)))
                        # print(diversity_scores_of_unfair_tasks)
                unfair_tasks.append(this_pair_unfair)
        # dp_unfairness[dp] = f"{sum(unfair_tasks)} / {len(unfair_tasks)}"
        dp_unfairness[dp] = sum(unfair_tasks) / len(unfair_tasks)

        # Example list/array of scores
        if len(diversity_scores_of_unfair_tasks) > 0:
            dsut_scores = np.array(diversity_scores_of_unfair_tasks)
            dp_diversity[dp] = dsut_scores.mean()
        else:
            print(model, dp, f"{diversity_scores_of_unfair_tasks=}")
            dp_diversity[dp] = np.nan
    
    model_pvalues.append({
        "Model Name" : model,
        **dp_unfairness
    })
    
    model_significant_diversity_scores.append({
        "Model Name" : model,
        **dp_diversity
    })

demo_pvalue_perquestion = pd.DataFrame(model_pvalues)
sig_diversity_scores = pd.DataFrame(model_significant_diversity_scores)

/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:573: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


llama31_70B_Inst sex diversity_scores_of_unfair_tasks=[]
Qwen25_1dot5B_Inst sex diversity_scores_of_unfair_tasks=[]


In [5]:
def df_to_latex_percent(df):
    def format_val(val):
        if isinstance(val, float):
            return f"{round(val * 100, 1)}\\%"
        return str(val).replace("_", "\\_")

    formatted_df = df.copy()
    formatted_df.columns = [col.replace("_", "\\_") for col in df.columns]
    for col in formatted_df.columns:
        formatted_df[col] = formatted_df[col].apply(format_val)

    return formatted_df.to_latex(index=False, escape=False)

In [6]:
print(df_to_latex_percent(demo_pvalue_perquestion))

\begin{tabular}{llllllll}
\toprule
Model Name & employment\_sector & race & education & sex & age & language & nationality\_group \\
\midrule
gpt\_4o & 7.7\% & 17.9\% & 7.7\% & 5.1\% & 15.4\% & 5.1\% & 20.5\% \\
llama31\_70B\_Inst & 15.4\% & 23.1\% & 7.7\% & 0.0\% & 17.9\% & 10.3\% & 38.5\% \\
Qwen25\_72B\_Inst & 5.1\% & 10.3\% & 5.1\% & 7.7\% & 5.1\% & 7.7\% & 23.1\% \\
llama31\_8B\_Inst & 20.5\% & 17.9\% & 5.1\% & 5.1\% & 20.5\% & 12.8\% & 30.8\% \\
Qwen25\_7B\_Inst & 12.8\% & 23.1\% & 25.6\% & 7.7\% & 10.3\% & 15.4\% & 28.2\% \\
llama32\_1B\_Inst & 7.7\% & 23.1\% & 2.6\% & 5.1\% & 7.7\% & 2.6\% & 30.8\% \\
Qwen25\_1dot5B\_Inst & 20.5\% & 17.9\% & 5.1\% & 0.0\% & 15.4\% & 23.1\% & 30.8\% \\
\bottomrule
\end{tabular}



In [7]:
print(df_to_latex_percent(sig_diversity_scores))

\begin{tabular}{llllllll}
\toprule
Model Name & employment\_sector & race & education & sex & age & language & nationality\_group \\
\midrule
gpt\_4o & 13.0\% & 11.3\% & 5.0\% & 3.7\% & 6.8\% & 10.3\% & 6.3\% \\
llama31\_70B\_Inst & 5.1\% & 8.7\% & 11.7\% & nan\% & 8.7\% & 8.4\% & 13.4\% \\
Qwen25\_72B\_Inst & 5.5\% & 14.4\% & 4.4\% & 10.7\% & 21.0\% & 16.3\% & 8.1\% \\
llama31\_8B\_Inst & 9.2\% & 8.9\% & 30.6\% & 9.2\% & 7.1\% & 12.4\% & 10.1\% \\
Qwen25\_7B\_Inst & 8.6\% & 6.4\% & 13.6\% & 6.3\% & 6.3\% & 6.3\% & 8.7\% \\
llama32\_1B\_Inst & 15.8\% & 9.7\% & 14.1\% & 7.7\% & 10.4\% & 13.3\% & 10.9\% \\
Qwen25\_1dot5B\_Inst & 8.3\% & 6.9\% & 19.1\% & nan\% & 5.7\% & 7.0\% & 7.3\% \\
\bottomrule
\end{tabular}

